# 01 · Best Practices: testing & task outputs

Three questions the Manas team raised on Slack:
> 1. **Testing**: how do people ensure that prod tasks run without user errors and recover well from system errors?
> 2. **Infrastructure**: how do other Flyte users track / aggregate / use task outputs? Database? Direct Flyte API query? Both?
> 3. **Integration**: do companies often wrap Flyte with internal tools, or do they let users use the Flyte tools directly?

This notebook digs into **1 (testing)** and **2 (task outputs)** with the example
pipeline in [`../workflows/01_best_practices.py`](../workflows/01_best_practices.py).
Question 3 (integration) is covered in the session slides.

Docs: [caching](https://www.union.ai/docs/v2/flyte/user-guide/core-concepts/caching/) ·
[errors & retries](https://www.union.ai/docs/v2/flyte/user-guide/core-concepts/task-configuration/) ·
[flyte.remote](https://www.union.ai/docs/v2/flyte/user-guide/development-cycle/inspecting-runs/)

### Connect

`flyte.init_from_config` opens the connection every later `flyte.run` targets.
This notebook points at `.flyte/config.yaml`.

In [ ]:
from pathlib import Path
import flyte

flyte.init_from_config(Path(".flyte") / "config.yaml")

## Question 1 · Testing

Two very different failure classes hide under "testing":

- **User errors**: bad input, a logic bug. You want these *caught in a test* or
  *tolerated at runtime*, not silently corrupting a batch.
- **System errors**: a pod dies, an API rate-limits, the network blips. You want
  these *recovered automatically*.

Flyte gives you **two ways to invoke a task in a test**
([docs](https://www.union.ai/docs/v2/union/user-guide/tasks/task-programming/unit-testing/)):

- **Direct invocation**: call the task like a normal function (sync) or `await` it
  (async). This *bypasses* Flyte (no serialization/caching/type checks), so it's the
  fast way to test business logic. **No cluster.**
- **`flyte.run(...)`**: engages the full machinery (serialization, type checks,
  caching); use it to test the *serialized* path and typed outputs.

First we build the pipeline, then we test it both ways.

In [13]:
from datetime import timedelta
import asyncio, random
from pydantic import BaseModel

worker_env = flyte.TaskEnvironment(
    name="best_practices_worker",
    resources=flyte.Resources(cpu=1, memory="500Mi"),
)
driver_env = flyte.TaskEnvironment(
    name="best_practices_driver",
    resources=flyte.Resources(cpu=1, memory="500Mi"),
    depends_on=[worker_env],  # the driver calls worker tasks
)

# A typed result: Flyte serializes and durably persists it (Question 2).
class Report(BaseModel):
    scored: int
    skipped: int
    total_score: int

# Plain business logic: NO Flyte. This is what we unit-test.
def score(record: dict) -> int:
    if "value" not in record:
        raise ValueError(f"record {record.get('id', '?')} is missing 'value'")
    return int(record["value"]) * int(record.get("weight", 1))

**System errors → task configuration.** `cache="auto"` skips re-computation
on identical inputs, `retries` re-runs transient failures, and `timeout` kills a
hung attempt. Together they recover from the "rate limit / network blip" class with
no custom code. (Flyte can't tell a user error from a system one, so a deterministic
`ValueError` will also burn its retries before surfacing, validate inputs early
when that matters.)

In [14]:
@worker_env.task(cache="auto", retries=2, timeout=timedelta(minutes=2))
async def score_record(record: dict, flaky: bool = False) -> int:
    if flaky and random.random() < 0.5:
        raise ConnectionError("transient upstream 503, Flyte retries this attempt")
    return score(record)

**User errors → tolerate at fan-out.** The driver fans out one action per record
with `asyncio.gather`. `return_exceptions=True` hands the bad record's `ValueError`
back as a *value*, so we skip it and the run still succeeds.

> `flyte.map(score_record, records, return_exceptions=True)` is the equivalent
> mapping idiom when you'd rather have a single map node than a gather.

In [15]:
@driver_env.task
async def main(flaky: bool = False) -> Report:
    records = [
        {"id": 1, "value": 10, "weight": 2},
        {"id": 2, "value": 5},
        {"id": 3, "weight": 3},          # <- malformed on purpose
        {"id": 4, "value": 7, "weight": 4},
    ]
    with flyte.group("score-fanout"):
        results = await asyncio.gather(
            *[score_record(r, flaky=flaky) for r in records],
            return_exceptions=True,
        )
    scored = skipped = total = 0
    for rec, res in zip(records, results):
        if isinstance(res, Exception):
            print(f"skipping record {rec.get('id','?')}: {res}")
            skipped += 1
        else:
            scored += 1; total += res
    return Report(scored=scored, skipped=skipped, total_score=total)

**Method 1: direct invocation, no cluster.** We test the plain function *and*
the async task body. Awaiting an `@env.task` runs its body in-process, bypassing
Flyte machinery, instant and CI-friendly.

In [16]:
# business logic: plain function, called directly
assert score({"id": 1, "value": 10, "weight": 2}) == 20
try:
    score({"id": 3, "weight": 3})                 # malformed -> user error
except ValueError as e:
    print("caught expected user error:", e)

# the async task body: `await` runs it in-process (no serialization/caching)
assert await score_record({"id": 1, "value": 10, "weight": 2}) == 20
print("direct-invocation tests passed")

caught expected user error: record 3 is missing 'value'
direct-invocation tests passed


**Method 2: `flyte.run(...)`, the serialized path.** Direct invocation skips
Flyte; to test the task *as it runs in production*, submit it. `flyte.run`
serializes the inputs, runs each task in its container, and hands back the typed
`Report`, so this exercises type-checking and the driver → worker wiring, not just
the Python body.

In [17]:
run = flyte.run(main)                          # full machinery, on the cluster
print("Run URL:", run.url)
run.wait()
report = run.outputs()
print("serialized output:", report)
assert report.o0.scored == 3 and report.o0.skipped == 1   # one malformed record skipped

> Building 2 images...

> Building image flyte for environment best_practices_driver

> Building image flyte for environment best_practices_worker

✓ Built image for environment best_practices_driver: ghcr.io/flyteorg/flyte:py3.13-v2.6.1

✓ Built image for environment best_practices_worker: ghcr.io/flyteorg/flyte:py3.13-v2.6.1

Run URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/leon-demo/runs/u4z2l9vv9vsrz2672h92


Run 'u4z2l9vv9vsrz2672h92' completed successfully.

serialized output: ActionOutputs(o0=scored=3 skipped=1 total_score=53)


The same tests live as a pytest suite in
[`../workflows/test_best_practices.py`](../workflows/test_best_practices.py),
run `pytest test_best_practices.py`. The direct-invocation tests need no cluster;
the `flyte.run(...)` serialized-path test is marked for when one is configured.

## Question 2 · Infrastructure: tracking & aggregating outputs

You don't need a database to *hold* outputs. Flyte offloads every task's return
value to object storage and records it against the run, so you can fetch a typed
result two ways. We'll use the "hello, Union" workflow from session 1 as the example.

In [18]:
# The hello-world workflow from session 1.
hello_env = flyte.TaskEnvironment(name="hello", resources=flyte.Resources(cpu=1, memory="250Mi"))

@hello_env.task
def join_names(first_name: str, last_name: str) -> str:
    return f"{first_name} {last_name}"

@hello_env.task
async def get_name_length(name: str) -> int:
    return len(name)

@hello_env.task
async def hello(first_name: str = "Ada", last_name: str = "Lovelace") -> str:
    full = join_names(first_name, last_name)
    return f"'{full}' has {await get_name_length(full)} characters"

# (1) From the run handle, right after launching:
run = flyte.run(hello, first_name="Ada", last_name="Lovelace")
run.wait()
print("from flyte.run:      ", run.outputs())

> Building 1 image...

> Building image flyte for environment hello

✓ Built image for environment hello: ghcr.io/flyteorg/flyte:py3.13-v2.6.1

Run 'umgmfk82fmm9dwqdfgps' completed successfully.

from flyte.run:       ActionOutputs(o0="'Ada Lovelace' has 12 characters")


**(2) Later, from anywhere**, re-fetch the same run *by name* with the
`flyte.remote` API, no handle required. This is the hook an aggregator or dashboard
uses: query Flyte as the system of record, then *sync* results into your own store.

In [19]:
import flyte.remote

past = flyte.remote.Run.get(name=run.name)     # any script can re-fetch by name
print("phase:              ", past.phase)
print("from flyte.remote:  ", past.outputs())  # same typed result, straight from the API

phase:               ActionPhase.SUCCEEDED
from flyte.remote:   ActionOutputs(o0="'Ada Lovelace' has 12 characters")


So: **`run.outputs()`** for the run you just launched,
**`flyte.remote.Run.get(name).outputs()`** for any past run, and
`flyte.remote.Action.listall(for_run_name=...)` to walk the individual task
actions. Flyte is the system of record; wrap those calls in a nightly job to
populate a warehouse or dashboard.

**Next:** [02 · Developing agentic pipelines on Union](./02_agentic_pipeline.ipynb).